# Exercise 02 — Save or Let Go Starter Notebook

> **Graded, multiweek exercise.** This is not the short in-class Guided Practice notebook. Save your own copy and continue it across Weeks 4–6.

The exercise progresses through the same retention decision:

1. **Week 4 checkpoint — build the probability model:** inspect relationships; fit one-input, two-input, and full-feature logistic models; validate the baseline.
2. **Week 5 checkpoint A — regularize the model:** redo the full model with Ridge and Lasso while holding rows, preprocessing, folds, and metrics fixed.
3. **Week 5 checkpoint B — build the policy:** use campaign economics, contact capacity, and sensitivity analysis to choose a threshold.
4. **Week 6 checkpoint — compare and recommend:** add a tree-based candidate and finalize the model-and-policy recommendation.

Use **File → Save a copy in Drive** before editing. Stop at each checkpoint and record your reasoning, not only the output.


## Before you run anything

**Prediction question:** What is the probability that a customer will churn?

**Decision question:** Which customers should receive a retention contact, given limited capacity and campaign costs?

The model produces a probability. A threshold and the campaign assumptions turn that probability into an action.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42


## Load the data

The loader first looks for a local repository checkout. In Colab, it reads the same files directly from the public GitHub repository, so no authorization step should be required.


In [ ]:
LOCAL_DATA = Path("../data")
RAW_BASE = "https://raw.githubusercontent.com/lathrahul/modern-ai-ml-exercises/main/exercise-02-retention/data"

def read_course_csv(filename):
    local_file = LOCAL_DATA / filename
    source = local_file if local_file.exists() else f"{RAW_BASE}/{filename}"
    return pd.read_csv(source)

train = read_course_csv("train.csv")
test = read_course_csv("test.csv")
costs = read_course_csv("campaign_costs.csv")

print("train:", train.shape, "test:", test.shape)
train.head()


# Week 4 checkpoint — Look first

Correlation is a descriptive starting point. It tells us whether two numeric variables move together in this sample. It does **not** produce a churn probability, control for other variables, or establish causality.

Because `churned` is coded 0/1, its Pearson correlation with a numeric feature is also the point-biserial correlation.


In [ ]:
target = "churned"
numeric_for_corr = ["tenure_months", "monthly_charges", "total_charges", "senior_citizen", target]
corr = train[numeric_for_corr].corr()

display(corr.round(2))

fig, ax = plt.subplots(figsize=(7, 5.5))
image = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=35, ha="right")
ax.set_yticks(range(len(corr.index)), corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=10)
ax.set_title("Numeric correlations: a first look, not a fitted model")
plt.colorbar(image, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()


### Discuss 1 — read the correlations

1. Which numeric variable has the strongest relationship with churn?
2. What can this table tell us?
3. What can it **not** tell us that logistic regression can?


In [ ]:
contract_rates = (
    train.groupby("contract", observed=True)[target]
    .agg(churn_rate="mean", customers="size")
    .sort_values("churn_rate")
)
display(contract_rates.style.format({"churn_rate": "{:.1%}"}))

ax = contract_rates["churn_rate"].plot(kind="barh", color="#2a63ad", figsize=(7, 3.8))
ax.set_xlabel("Observed churn rate")
ax.set_ylabel("Contract")
ax.set_title("A categorical relationship: churn rate by contract")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
plt.tight_layout()
plt.show()


# Week 4 checkpoint — Build ordinary logistic regression slowly

**Goal:** see what logistic regression adds beyond correlation, then finish with a working baseline you can evaluate.

We will compare one-input, two-input, and full-feature models; interpret coefficients as changes in odds; evaluate all models on the same validation rows; and use stratified K-fold cross-validation rather than depending on one split.


## 1. Prevalence and the simple baseline

If the churn rate is low, a model can have high accuracy by predicting that everyone stays. That is why accuracy alone is not enough.


In [ ]:
target = "churned"
prevalence = train[target].mean()
always_stay_accuracy = 1 - prevalence

summary = pd.Series({
    "churn prevalence": prevalence,
    "always-stay accuracy": always_stay_accuracy,
})
display(summary.to_frame("value").style.format("{:.1%}"))

ax = summary.plot(kind="bar", color=["#2a63ad", "#b54740"], ylim=(0, 1), rot=0)
ax.set_ylabel("Share of customers")
ax.set_title("Class prevalence and the always-stay baseline")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1%}", (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.show()


### Discuss 2 — establish the baseline

1. What would accuracy report for the always-stay rule?
2. What does that number fail to tell us about customers who actually churn?


## 2. Train, validation, and test are different roles

- **Training rows** are used to learn preprocessing and coefficients.
- **Validation rows** estimate performance and help us compare choices.
- **Test rows** should remain untouched until the policy is finalized.

Stratification keeps approximately the same churn prevalence on both sides of this first split.


In [ ]:
X = train.drop(columns=[target, "customer_id"])
y = train[target]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)

pd.DataFrame({
    "rows": [len(y_train), len(y_valid)],
    "churn rate": [y_train.mean(), y_valid.mean()],
}, index=["train", "validation"])


## 3. Model A — one input

Start with `tenure_months`. Before fitting, predict the coefficient sign:

- positive means longer tenure raises the odds of churn;
- negative means longer tenure lowers the odds of churn.

The odds multiplier for one additional month is $e^{β}$.


In [ ]:
one_features = ["tenure_months"]

one_input_model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(C=1e6, max_iter=2000, random_state=RANDOM_STATE)),
])
one_input_model.fit(X_train[one_features], y_train)

beta_tenure = one_input_model.named_steps["model"].coef_[0, 0]
print(f"tenure coefficient: {beta_tenure:.3f}")
print(f"odds multiplier for one additional month: {np.exp(beta_tenure):.3f}")

tenure_grid = pd.DataFrame({"tenure_months": np.arange(0, 73)})
tenure_probability = one_input_model.predict_proba(tenure_grid)[:, 1]
plt.figure(figsize=(7, 4))
plt.plot(tenure_grid["tenure_months"], tenure_probability, color="#2a63ad", linewidth=3)
plt.xlabel("Tenure in months")
plt.ylabel("Predicted probability of churn")
plt.ylim(0, 1)
plt.title("Model A: one input produces a probability curve")
plt.grid(alpha=0.2)
plt.show()


### Discuss 3 — coefficient, odds, probability

1. Does the coefficient sign match the correlation sign?
2. Does the odds multiplier describe odds or probability?
3. Is the change in probability constant across the whole curve?


## 4. Model B — add a second input

Now add `monthly_charges`. Each coefficient answers a conditional question: how does one input relate to churn **while the other included input stays fixed**?


In [ ]:
two_features = ["tenure_months", "monthly_charges"]
two_input_model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(C=1e6, max_iter=2000, random_state=RANDOM_STATE)),
])
two_input_model.fit(X_train[two_features], y_train)

two_coef = pd.DataFrame({
    "coefficient": two_input_model.named_steps["model"].coef_.ravel(),
}, index=two_features)
two_coef["odds multiplier for one unit"] = np.exp(two_coef["coefficient"])
display(two_coef.round(3))

example_customers = pd.DataFrame({
    "tenure_months": [6, 6, 48, 48],
    "monthly_charges": [35, 90, 35, 90],
})
example_customers["predicted churn probability"] = two_input_model.predict_proba(example_customers)[:, 1]
display(example_customers.style.format({"predicted churn probability": "{:.1%}"}))


### Discuss 4 — what changed after adding a variable?

1. Compare customers with the same tenure and different monthly charges.
2. Compare customers with the same monthly charges and different tenure.
3. Why can a coefficient change after another variable enters the model?


## 5. Model C — use the full feature set safely

Scaling, imputation, and category encoding learn values from data. Keeping them inside the pipeline ensures that each fit learns only from its training rows.


In [ ]:
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_features),
    ("categorical", categorical_pipe, categorical_features),
])

baseline_model = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(C=1e6, max_iter=2000, random_state=RANDOM_STATE)),
])

baseline_model


## 6. Compare Models A, B, and C on the same validation customers

The comparison changes only the feature set. The validation rows and metrics stay fixed.


In [ ]:
baseline_model.fit(X_train, y_train)
valid_probability = baseline_model.predict_proba(X_valid)[:, 1]

one_probability = one_input_model.predict_proba(X_valid[one_features])[:, 1]
two_probability = two_input_model.predict_proba(X_valid[two_features])[:, 1]

def probability_metrics(actual, probability):
    prediction = (probability >= 0.50).astype(int)
    return {
        "accuracy @ 0.50": accuracy_score(actual, prediction),
        "log loss": log_loss(actual, probability),
        "ROC-AUC": roc_auc_score(actual, probability),
        "PR-AUC": average_precision_score(actual, probability),
    }

model_comparison = pd.DataFrame({
    "Model A: tenure only": probability_metrics(y_valid, one_probability),
    "Model B: tenure + charges": probability_metrics(y_valid, two_probability),
    "Model C: full feature set": probability_metrics(y_valid, valid_probability),
}).T
display(model_comparison.round(3))

metrics = pd.Series({
    "log loss (lower is better)": log_loss(y_valid, valid_probability),
    "ROC-AUC": roc_auc_score(y_valid, valid_probability),
    "PR-AUC": average_precision_score(y_valid, valid_probability),
})
display(metrics.to_frame("validation result").round(3))

threshold = 0.50
valid_prediction = (valid_probability >= threshold).astype(int)
cm = confusion_matrix(y_valid, valid_prediction)

fig, ax = plt.subplots(figsize=(4.8, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], labels=["Predict stay", "Predict churn"])
ax.set_yticks([0, 1], labels=["Actually stay", "Actually churn"])
ax.set_title(f"Confusion matrix at threshold {threshold:.2f}")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14,
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.show()


### Discuss 5 — evaluate the baseline model

1. Which metric answers “Are the probabilities numerically useful?”
2. Which error in the confusion matrix represents a customer who churned but was not contacted?
3. Change `threshold` from `0.50` to `0.30` and rerun only the confusion-matrix cell. What moves?


## 7. Why cross-validation?

One validation split may be unusually easy or difficult. In five-fold cross-validation, each fold serves as validation once while the other four folds train the complete pipeline.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "neg_log_loss": "neg_log_loss",
}

cv_result = cross_validate(baseline_model, X, y, cv=cv, scoring=scoring)
fold_results = pd.DataFrame({
    "ROC-AUC": cv_result["test_roc_auc"],
    "PR-AUC": cv_result["test_pr_auc"],
    "log loss": -cv_result["test_neg_log_loss"],
}, index=[f"fold {i}" for i in range(1, 6)])

display(fold_results.round(3))
display(pd.DataFrame({"mean": fold_results.mean(), "std": fold_results.std()}).round(3))


### Week 4 checkpoint — explain the complete baseline

In two or three sentences, record:

- what correlation showed before fitting a model;
- what changed from Model A to Model B to Model C;
- why the five-fold mean and spread are more informative than one validation score.


# Bridge to Week 5 — two meanings of iteration

Models A, B, and C were **modeling iterations**: we changed the feature set and compared each specification on the same validation customers.

| Modeling iteration | Inputs | Validation log loss |
|---|---|---:|
| Model A | tenure | 0.503 |
| Model B | tenure + monthly charges | 0.437 |
| Model C | full feature set | 0.399 |

Lower validation log loss means the probabilities were numerically better on these validation rows. Model C did not win because its solver simply ran for more steps.

Within one fixed model, the solver also performs **optimizer iterations**. It repeatedly updates the coefficients to reduce the training objective. The next cell makes those updates visible for Model B.


In [ ]:
# A transparent batch-gradient-descent demonstration for Model B.
# This is for learning; scikit-learn uses a faster numerical solver.
iteration_features = ["tenure_months", "monthly_charges"]
iteration_scaler = StandardScaler()
X_iteration_train = iteration_scaler.fit_transform(X_train[iteration_features])
X_iteration_valid = iteration_scaler.transform(X_valid[iteration_features])

# Add a column of ones for the intercept.
X_iteration_train = np.column_stack([np.ones(len(X_iteration_train)), X_iteration_train])
X_iteration_valid = np.column_stack([np.ones(len(X_iteration_valid)), X_iteration_valid])

beta = np.zeros(X_iteration_train.shape[1])
learning_rate = 0.50
checkpoints = [0, 1, 2, 5, 10, 20, 50, 100, 200]
history = []

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

for iteration in range(201):
    if iteration in checkpoints:
        train_p = sigmoid(X_iteration_train @ beta)
        valid_p = sigmoid(X_iteration_valid @ beta)
        history.append({
            "optimizer iteration": iteration,
            "training log loss": log_loss(y_train, train_p),
            "validation log loss": log_loss(y_valid, valid_p),
        })
    if iteration < 200:
        train_p = sigmoid(X_iteration_train @ beta)
        gradient = X_iteration_train.T @ (train_p - y_train.to_numpy()) / len(y_train)
        beta -= learning_rate * gradient

iteration_history = pd.DataFrame(history)
display(iteration_history.round(3))

ax = iteration_history.plot(
    x="optimizer iteration",
    y=["training log loss", "validation log loss"],
    marker="o",
    color=["#2a63ad", "#e87832"],
    figsize=(8, 4.5),
)
ax.set_ylabel("Log loss (lower is better)")
ax.set_title("Optimizer iterations reduce loss until the fit converges")
ax.grid(alpha=0.2)
plt.show()


### Read the iteration curve

- Early coefficient updates reduce both training and validation log loss in this example.
- The improvements become very small as the solver converges.
- `max_iter` sets a ceiling on solver work. Increase it when the model has not converged; do not treat it as a generalization control.
- Validation log loss is not guaranteed to improve whenever training log loss improves.
- Regularization comes next because it changes the objective and the final coefficient solution, especially when the full feature set contains correlated signals.


# Week 5 checkpoint A — Redo Model C with Ridge and Lasso

**Goal:** compare three penalty choices while keeping the feature set, split logic, preprocessing, and metrics fixed.

- The near-unpenalized model provides the baseline.
- **Ridge (L2)** shrinks correlated signals and usually keeps all coefficients.
- **Lasso (L1)** can set some coefficients exactly to zero.

`C` is the inverse of regularization strength in scikit-learn: a smaller `C` means stronger shrinkage.


In [ ]:
models = {
    "near-unpenalized": LogisticRegression(C=1e6, max_iter=2000, random_state=RANDOM_STATE),
    "Ridge (L2)": LogisticRegression(penalty="l2", C=0.20, max_iter=2000, random_state=RANDOM_STATE),
    "Lasso (L1)": LogisticRegression(penalty="l1", solver="liblinear", C=0.20, max_iter=2000, random_state=RANDOM_STATE),
}

comparison_rows = []
fitted_pipelines = {}

for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
    result = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
    comparison_rows.append({
        "model": name,
        "ROC-AUC mean": result["test_roc_auc"].mean(),
        "ROC-AUC std": result["test_roc_auc"].std(),
        "PR-AUC mean": result["test_pr_auc"].mean(),
        "log loss mean": -result["test_neg_log_loss"].mean(),
    })
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

comparison = pd.DataFrame(comparison_rows).set_index("model")
display(comparison.round(3))

print("Read this as a redo of Model C: the inputs and evaluation stay fixed; only the penalty changes.")


## Inspect what happened to the coefficients

The chart compares the largest coefficients after preprocessing. A positive coefficient increases log-odds of churn; a negative coefficient decreases them, holding the other encoded inputs fixed.


In [ ]:
feature_names = fitted_pipelines["near-unpenalized"].named_steps["preprocess"].get_feature_names_out()
coef_frame = pd.DataFrame(index=feature_names)

for name, pipe in fitted_pipelines.items():
    coef_frame[name] = pipe.named_steps["model"].coef_.ravel()

top_features = coef_frame["near-unpenalized"].abs().nlargest(12).index
plot_data = coef_frame.loc[top_features].sort_values("near-unpenalized")
plot_data.plot(kind="barh", figsize=(9, 6), color=["#b54740", "#2a63ad", "#27835f"])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Coefficient on the log-odds scale")
plt.title("How Ridge and Lasso change the fitted coefficients")
plt.tight_layout()
plt.show()

zero_counts = (coef_frame.abs() < 1e-10).sum().rename("coefficients equal to zero")
zero_counts.to_frame()


### Week 5 checkpoint A — compare the redo

Record short answers:

1. Which model produces the most exact zeros?
2. Did regularization materially change mean validation performance?
3. If two models perform similarly, what practical reason might favor Ridge, Lasso, or the simpler baseline?

Do not claim that a zero Lasso coefficient proves a feature is unimportant. Correlated features can exchange which signal is retained.


# Week 5 checkpoint B — From churn probabilities to a retention policy

The model estimates risk. The operating policy decides whom to contact. Use the supplied campaign assumptions, a contact-capacity limit, and sensitivity analysis rather than selecting a threshold from accuracy alone.

For this checkpoint, use the Ridge validation probabilities as a defensible working candidate. You may replace the model later if your Week 6 comparison supports a different choice.


In [ ]:
# Use validation probabilities from the already-fitted Ridge model.
ridge_probability = fitted_pipelines["Ridge (L2)"].predict_proba(X_valid)[:, 1]

assumptions = costs.set_index("parameter")["value_usd_or_rate"].to_dict()
contact_cost = assumptions["contact_cost"]
offer_cost = assumptions["offer_cost_if_accepted"]
gross_margin_saved = assumptions["gross_margin_saved"]
acceptance_rate = assumptions["offer_acceptance_rate"]
success_rate = assumptions["intervention_success_rate"]

# A planning constraint: the team can contact at most 20% of validation customers.
capacity = int(np.floor(0.20 * len(y_valid)))

def threshold_table(probability, actual, margin_multiplier=1.0):
    rows = []
    for threshold in np.arange(0.10, 0.81, 0.05):
        contact = probability >= threshold
        true_churn_contacted = ((actual.to_numpy() == 1) & contact).sum()
        expected_saved = true_churn_contacted * acceptance_rate * success_rate * gross_margin_saved * margin_multiplier
        expected_spend = contact.sum() * (contact_cost + acceptance_rate * offer_cost)
        rows.append({
            "threshold": threshold,
            "contacts": int(contact.sum()),
            "within capacity": bool(contact.sum() <= capacity),
            "expected campaign value": expected_saved - expected_spend,
        })
    return pd.DataFrame(rows)

policy = threshold_table(ridge_probability, y_valid)
display(policy.style.format({"threshold": "{:.2f}", "expected campaign value": "${:,.0f}"}))

feasible = policy[policy["within capacity"]]
selected = feasible.loc[feasible["expected campaign value"].idxmax()]
print(f"Capacity: {capacity} contacts")
print(f"Best listed feasible threshold: {selected['threshold']:.2f}")


## Week 5 sensitivity check

Campaign assumptions are estimates. Recalculate the policy if the saved margin is only 40% of the planning value. If the selected threshold changes, the policy is sensitive to that assumption and should be monitored closely.


In [ ]:
sensitivity = threshold_table(ridge_probability, y_valid, margin_multiplier=0.40)
display(sensitivity.style.format({"threshold": "{:.2f}", "expected campaign value": "${:,.0f}"}))

feasible_sensitivity = sensitivity[sensitivity["within capacity"]]
selected_sensitivity = feasible_sensitivity.loc[feasible_sensitivity["expected campaign value"].idxmax()]
print(f"Best listed feasible threshold under lower saved margin: {selected_sensitivity['threshold']:.2f}")


# Week 6 checkpoint — Compare nonlinear candidates before trees

Keep the decision problem and evidence design fixed. Reuse the same training and validation customers, preprocessing pipeline, metrics, campaign assumptions, and capacity limit. Change only the model family.

The first checkpoint compares three ways to relax a straight-line log-odds assumption:

- a quadratic polynomial for tenure;
- a spline for tenure, selected from a small declared configuration set;
- a GAM-style additive logistic model with separate smooth functions for tenure and monthly charges.

Carry one nonlinear candidate into the later comparison with a shallow tree and Random Forest. Do not use the final test customers to select model settings or thresholds.


## 0. Nonlinear benchmark — polynomial, spline, and GAM-style models

Keep the rows, folds, other features, regularization, and metrics fixed. Change how the model represents nonlinear relationships.

The candidates answer different questions:

- **Quadratic polynomial:** does one global bend improve the tenure relationship?
- **Spline:** do local bends improve fit without unstable boundary behavior?
- **GAM-style additive logistic:** does smoothing both tenure and monthly charges help while preserving an additive explanation?

All three remain logistic probability models. The GAM-style implementation uses spline bases plus L2 regularization; it does not use a specialized GAM smoothness penalty.


In [ ]:
other_numeric_features = [c for c in numeric_features if c != "tenure_months"]

def summarize_probability_model(name, pipeline):
    candidate_cv = cross_validate(pipeline, X, y, cv=cv, scoring=scoring)
    pipeline.fit(X_train, y_train)
    probability = pipeline.predict_proba(X_valid)[:, 1]
    summary = {
        "candidate": name,
        "ROC-AUC mean": candidate_cv["test_roc_auc"].mean(),
        "ROC-AUC std": candidate_cv["test_roc_auc"].std(),
        "PR-AUC mean": candidate_cv["test_pr_auc"].mean(),
        "log loss mean": -candidate_cv["test_neg_log_loss"].mean(),
        "held-out log loss": log_loss(y_valid, probability),
    }
    return summary, probability

polynomial_preprocess = ColumnTransformer([
    ("tenure_polynomial", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
        ("scale", StandardScaler()),
    ]), ["tenure_months"]),
    ("numeric", numeric_pipe, other_numeric_features),
    ("categorical", categorical_pipe, categorical_features),
])

polynomial_pipeline = Pipeline([
    ("preprocess", polynomial_preprocess),
    ("model", LogisticRegression(
        penalty="l2", C=0.20, max_iter=2000, random_state=RANDOM_STATE
    )),
])

polynomial_summary, polynomial_probability = summarize_probability_model(
    "quadratic tenure", polynomial_pipeline
)

spline_specs = {
    "low: 3 knots, degree 1": {"n_knots": 3, "degree": 1},
    "moderate: 5 knots, degree 3": {"n_knots": 5, "degree": 3},
    "high: 9 knots, degree 3": {"n_knots": 9, "degree": 3},
}

fitted_spline_pipelines = {}
spline_probabilities = {}
spline_rows = []

for spline_name, spec in spline_specs.items():
    spline_preprocess = ColumnTransformer([
        ("tenure_spline", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("spline", SplineTransformer(
                n_knots=spec["n_knots"],
                degree=spec["degree"],
                include_bias=False,
            )),
            ("scale", StandardScaler()),
        ]), ["tenure_months"]),
        ("numeric", numeric_pipe, other_numeric_features),
        ("categorical", categorical_pipe, categorical_features),
    ])

    candidate = Pipeline([
        ("preprocess", spline_preprocess),
        ("model", LogisticRegression(
            penalty="l2", C=0.20, max_iter=2000, random_state=RANDOM_STATE
        )),
    ])

    candidate_summary, candidate_probability = summarize_probability_model(spline_name, candidate)

    fitted_spline_pipelines[spline_name] = candidate
    spline_probabilities[spline_name] = candidate_probability
    candidate_summary["basis functions"] = spec["n_knots"] + spec["degree"] - 1
    spline_rows.append(candidate_summary)

spline_configuration_results = (
    pd.DataFrame(spline_rows)
    .set_index("candidate")
    .sort_values("log loss mean")
)
display(spline_configuration_results.round(3))

selected_spline_name = spline_configuration_results.index[0]
spline_pipeline = fitted_spline_pipelines[selected_spline_name]
spline_probability = spline_probabilities[selected_spline_name]
spline_summary = spline_configuration_results.loc[selected_spline_name, [
    "ROC-AUC mean", "ROC-AUC std", "PR-AUC mean", "log loss mean"
]]
print("Spline carried forward:", selected_spline_name)


### GAM-style additive logistic candidate

This candidate gives tenure and monthly charges separate smooth functions, then adds their contributions on the log-odds scale. Additivity keeps the component effects readable. It also means the model will miss an interaction unless one is added explicitly.


In [ ]:
gam_other_numeric = [
    c for c in numeric_features if c not in ["tenure_months", "monthly_charges"]
]

gam_preprocess = ColumnTransformer([
    ("tenure_smooth", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("spline", SplineTransformer(n_knots=5, degree=3, include_bias=False)),
        ("scale", StandardScaler()),
    ]), ["tenure_months"]),
    ("charges_smooth", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("spline", SplineTransformer(n_knots=5, degree=3, include_bias=False)),
        ("scale", StandardScaler()),
    ]), ["monthly_charges"]),
    ("numeric", numeric_pipe, gam_other_numeric),
    ("categorical", categorical_pipe, categorical_features),
])

gam_pipeline = Pipeline([
    ("preprocess", gam_preprocess),
    ("model", LogisticRegression(
        penalty="l2", C=0.20, max_iter=2000, random_state=RANDOM_STATE
    )),
])

gam_summary, gam_probability = summarize_probability_model(
    "GAM-style: smooth tenure + charges", gam_pipeline
)

selected_spline_summary = {
    "candidate": f"spline: {selected_spline_name}",
    **spline_summary.to_dict(),
    "held-out log loss": spline_configuration_results.loc[
        selected_spline_name, "held-out log loss"
    ],
}

nonlinear_results = (
    pd.DataFrame([polynomial_summary, selected_spline_summary, gam_summary])
    .set_index("candidate")
    .sort_values("log loss mean")
)
display(nonlinear_results.round(3))

nonlinear_candidates = {
    "quadratic tenure": (polynomial_pipeline, polynomial_probability),
    f"spline: {selected_spline_name}": (spline_pipeline, spline_probability),
    "GAM-style: smooth tenure + charges": (gam_pipeline, gam_probability),
}

selected_nonlinear_name = nonlinear_results.index[0]
nonlinear_pipeline, nonlinear_probability = nonlinear_candidates[selected_nonlinear_name]
nonlinear_summary = nonlinear_results.loc[selected_nonlinear_name, [
    "ROC-AUC mean", "ROC-AUC std", "PR-AUC mean", "log loss mean"
]]
print("Nonlinear candidate carried into the tree comparison:", selected_nonlinear_name)


### Worked example — compare shape and data support

Hold the other inputs at one real customer's values. Compare the models as tenure changes, then inspect the GAM-style monthly-charges component. The lower panels show where the training data support each curve.

These curves are conditional model views, not causal effects. Holding correlated features fixed can create hypothetical combinations that are uncommon in the observed data.


In [ ]:
tenure_grid = np.linspace(0, 72, 73)
reference_customer = X_train.iloc[[0]].copy()
curve_rows = pd.concat([reference_customer] * len(tenure_grid), ignore_index=True)
curve_rows["tenure_months"] = tenure_grid

fig, axes = plt.subplots(
    2, 2, figsize=(12, 7),
    gridspec_kw={"height_ratios": [3, 1]}
)

linear_curve = fitted_pipelines["Ridge (L2)"].predict_proba(curve_rows)[:, 1]
axes[0, 0].plot(
    tenure_grid, linear_curve,
    label="Ridge logistic",
    color="#2458A6", linestyle="--", linewidth=2,
)

tenure_models = {
    "Quadratic": polynomial_pipeline,
    "Selected spline": spline_pipeline,
    "GAM-style": gam_pipeline,
}
for color, (name, candidate) in zip(["#6F9ECF", "#0A8F8F", "#B7791F"], tenure_models.items()):
    axes[0, 0].plot(
        tenure_grid, candidate.predict_proba(curve_rows)[:, 1],
        label=name, color=color, linewidth=2.2,
    )

axes[0, 0].set_ylabel("Predicted churn probability")
axes[0, 0].set_title("Tenure representation")
axes[0, 0].legend(fontsize=9)
axes[1, 0].hist(
    X_train["tenure_months"], bins=np.arange(0, 79, 6),
    color="#D7DCE5", edgecolor="white"
)
axes[1, 0].set_xlabel("Tenure in months")
axes[1, 0].set_ylabel("Training rows")

charge_grid = np.linspace(
    X_train["monthly_charges"].quantile(0.01),
    X_train["monthly_charges"].quantile(0.99),
    100,
)
charge_rows = pd.concat([reference_customer] * len(charge_grid), ignore_index=True)
charge_rows["monthly_charges"] = charge_grid

axes[0, 1].plot(
    charge_grid,
    fitted_pipelines["Ridge (L2)"].predict_proba(charge_rows)[:, 1],
    label="Ridge logistic", color="#2458A6", linestyle="--", linewidth=2,
)
axes[0, 1].plot(
    charge_grid, gam_pipeline.predict_proba(charge_rows)[:, 1],
    label="GAM-style", color="#B7791F", linewidth=2.2,
)
axes[0, 1].set_title("Monthly-charges representation")
axes[0, 1].legend(fontsize=9)
axes[1, 1].hist(
    X_train["monthly_charges"], bins=18,
    color="#D7DCE5", edgecolor="white"
)
axes[1, 1].set_xlabel("Monthly charges")
axes[1, 1].set_ylabel("Training rows")

plt.tight_layout()
plt.show()


### Nonlinear checkpoint

1. Does the quadratic model capture the main bend, or does its global shape miss local behavior?
2. Does the selected spline improve mean log loss by more than fold-to-fold variation?
3. Does smoothing monthly charges in the GAM-style model add useful probability evidence?
4. Where do curves change in regions with few training rows?
5. Which one nonlinear candidate should proceed to the tree comparison, and what evidence could reverse that choice?


## 1. A shallow classification tree

A classification tree repeatedly splits the feature space into regions. Each leaf assigns a churn probability equal to the observed positive-class share in that leaf.

The constraints below are deliberately conservative. A deep tree can fit training noise and create leaves supported by very few customers.


In [ ]:
tree_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=40,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

tree_cv = cross_validate(tree_pipeline, X, y, cv=cv, scoring=scoring)
tree_pipeline.fit(X_train, y_train)
tree_probability = tree_pipeline.predict_proba(X_valid)[:, 1]

tree_summary = pd.Series({
    "ROC-AUC mean": tree_cv["test_roc_auc"].mean(),
    "ROC-AUC std": tree_cv["test_roc_auc"].std(),
    "PR-AUC mean": tree_cv["test_pr_auc"].mean(),
    "log loss mean": -tree_cv["test_neg_log_loss"].mean(),
})
tree_summary.to_frame("shallow tree").round(3)


### Worked example — read one tree as a rule path

After fitting, inspect the first two levels. For one validation customer, record:

1. the first decision rule;
2. the branch taken;
3. the final leaf probability;
4. the action at your current threshold;
5. one reason the path might be unstable after resampling.


In [ ]:
transformed_names = tree_pipeline.named_steps["preprocess"].get_feature_names_out()
tree_model = tree_pipeline.named_steps["model"]

plt.figure(figsize=(18, 7))
plot_tree(
    tree_model,
    feature_names=transformed_names,
    class_names=["stay", "churn"],
    max_depth=2,
    filled=True,
    rounded=True,
    fontsize=8,
)
plt.title("First two levels of the shallow churn tree")
plt.show()

leaf_support = pd.Series(tree_model.tree_.n_node_samples)
print(f"Tree depth: {tree_model.get_depth()}")
print(f"Number of leaves: {tree_model.get_n_leaves()}")
print(f"Smallest node support: {leaf_support.min()} training rows")


## 2. Tree types and when they answer different questions

| Tree type | Output | Typical use | Main caution |
|---|---|---|---|
| Classification tree | Class probability or label | Churn, default, fraud | Threshold and class imbalance remain policy choices |
| Regression tree | Numeric leaf mean | Spend, demand, value | Stepwise predictions can be unstable |
| Survival tree | Time-to-event risk | Time until churn or failure | Censoring and time origin must be correct |
| Uplift or causal tree | Treatment-effect difference | Who benefits from an intervention | Requires credible treatment-effect identification |

Save or Let Go is a classification problem. Its model predicts churn probability. The contact policy still requires a separate threshold and campaign assumptions.


## 3. Random Forest — average many decorrelated trees

Each forest tree receives a bootstrap sample and only a subset of candidate features at each split. Averaging helps when the trees make different errors.


In [ ]:
forest_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=7,
        min_samples_leaf=20,
        max_features="sqrt",
        class_weight="balanced_subsample",
        oob_score=True,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])

forest_cv = cross_validate(forest_pipeline, X, y, cv=cv, scoring=scoring, n_jobs=-1)
forest_pipeline.fit(X_train, y_train)
forest_probability = forest_pipeline.predict_proba(X_valid)[:, 1]
forest_model = forest_pipeline.named_steps["model"]

forest_summary = pd.Series({
    "ROC-AUC mean": forest_cv["test_roc_auc"].mean(),
    "ROC-AUC std": forest_cv["test_roc_auc"].std(),
    "PR-AUC mean": forest_cv["test_pr_auc"].mean(),
    "log loss mean": -forest_cv["test_neg_log_loss"].mean(),
    "OOB accuracy": forest_model.oob_score_,
})
forest_summary.to_frame("Random Forest").round(3)


### Out-of-bag evidence

OOB accuracy is a convenient internal check, but it does not replace the course validation design. Accuracy can also look strong when churn is uncommon. Keep the held-out probability metrics and policy evidence as the main comparison.


## 4. One comparison table for every candidate

The table below uses cross-validation results produced under the same folds and metrics. Lower log loss is better. Higher ROC-AUC and PR-AUC are better.


In [ ]:
week6_comparison = comparison.copy()
week6_comparison.loc[f"selected nonlinear: {selected_nonlinear_name}"] = nonlinear_summary[week6_comparison.columns]
week6_comparison.loc["shallow tree"] = tree_summary[week6_comparison.columns]
week6_comparison.loc["Random Forest"] = forest_summary[week6_comparison.columns]

display(week6_comparison.sort_values("log loss mean").round(3))

print("Candidate with lowest mean log loss:", week6_comparison["log loss mean"].idxmin())
print("Candidate with highest mean PR-AUC:", week6_comparison["PR-AUC mean"].idxmax())


### Checkpoint — model evidence

1. Does the tree or forest materially improve probability quality?
2. Is the improvement consistent across metrics and folds?
3. Does a readable tree provide enough performance even if the forest scores slightly better?
4. Which candidate should proceed to policy comparison, and what evidence could reverse that choice?


## 5. Compare policies, not only model scores

Recompute the threshold table for Ridge, the selected nonlinear candidate, the shallow tree, and the Random Forest. The selected model and threshold should be chosen together under the same capacity and campaign assumptions.


In [ ]:
candidate_probabilities = {
    "Ridge (L2)": ridge_probability,
    f"selected nonlinear: {selected_nonlinear_name}": nonlinear_probability,
    "shallow tree": tree_probability,
    "Random Forest": forest_probability,
}

policy_rows = []
candidate_policy_tables = {}

for model_name, probability in candidate_probabilities.items():
    model_policy = threshold_table(probability, y_valid)
    candidate_policy_tables[model_name] = model_policy
    feasible_policy = model_policy[model_policy["within capacity"]]
    best = feasible_policy.loc[feasible_policy["expected campaign value"].idxmax()]
    policy_rows.append({
        "model": model_name,
        "threshold": best["threshold"],
        "contacts": int(best["contacts"]),
        "expected campaign value": best["expected campaign value"],
    })

policy_comparison = pd.DataFrame(policy_rows).set_index("model")
display(policy_comparison.style.format({
    "threshold": "{:.2f}",
    "expected campaign value": "${:,.0f}",
}))


## 6. Subgroup and stability check

Use a business-relevant group that exists before the prediction moment. The example below compares recall for newer and established customers at each candidate's selected threshold. Treat small groups cautiously.


In [ ]:
tenure_group = pd.cut(
    X_valid["tenure_months"],
    bins=[-np.inf, 12, np.inf],
    labels=["newer: 0–12 months", "established: 13+ months"],
)

subgroup_rows = []
for model_name, probability in candidate_probabilities.items():
    threshold = float(policy_comparison.loc[model_name, "threshold"])
    prediction = probability >= threshold
    for group_name in tenure_group.cat.categories:
        mask = tenure_group == group_name
        actual_group = y_valid.loc[mask].to_numpy()
        predicted_group = prediction[mask.to_numpy()]
        positive = actual_group == 1
        recall = ((predicted_group == 1) & positive).sum() / positive.sum() if positive.sum() else np.nan
        subgroup_rows.append({
            "model": model_name,
            "group": str(group_name),
            "customers": int(mask.sum()),
            "positive cases": int(positive.sum()),
            "recall": recall,
        })

subgroup_results = pd.DataFrame(subgroup_rows)
display(subgroup_results.style.format({"recall": "{:.1%}"}))


### Stability prompt

Change the tree's `random_state` or one nearby depth/leaf constraint, rerun the tree, and record whether the first split, validation log loss, chosen threshold, or subgroup recall changes materially. Do not search many settings and report only the winner.


## 7. Permutation importance on validation data

Permutation importance asks how much validation performance falls after one original feature is shuffled. It describes what the fitted pipeline relies on. It does not show that changing the feature will change churn.


In [ ]:
forest_permutation = permutation_importance(
    forest_pipeline,
    X_valid,
    y_valid,
    scoring="neg_log_loss",
    n_repeats=8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance = (
    pd.DataFrame({
        "feature": X_valid.columns,
        "mean importance": forest_permutation.importances_mean,
        "std": forest_permutation.importances_std,
    })
    .sort_values("mean importance", ascending=False)
    .head(12)
)

display(importance.round(4))
importance.sort_values("mean importance").plot(
    x="feature", y="mean importance", kind="barh", legend=False, figsize=(8, 5), color="#2458A6"
)
plt.xlabel("Increase in validation log loss after shuffling")
plt.title("Random Forest permutation importance")
plt.tight_layout()
plt.show()


## 8. Final model-and-policy recommendation

Record all of the following before generating test predictions:

1. final model and why it earned selection;
2. threshold and contact capacity;
3. expected contacts and value per 10,000 customers;
4. sensitivity result under the lower saved-margin scenario;
5. subgroup or customer-experience concern;
6. one monitoring metric;
7. one condition under which the policy should not be used.

The final recommendation may retain Ridge if the tree-based candidates do not improve the decision enough to justify their complexity.


## 9. Create the final decision file only after the decision is recorded

Set `final_model` and `final_threshold` from the validation evidence above. Fit the selected pipeline on all labeled training data once, then generate probabilities for every test customer.


In [ ]:
# Example: replace these two lines only after recording the final decision.
final_model = forest_pipeline
final_threshold = float(policy_comparison.loc["Random Forest", "threshold"])

final_model.fit(X, y)
test_features = test.drop(columns="customer_id")
probabilities = final_model.predict_proba(test_features)[:, 1]

submission = pd.DataFrame({
    "customer_id": test["customer_id"],
    "churn_probability": probabilities,
    "contact_customer": (probabilities >= final_threshold).astype(int),
})

submission.to_csv("../submission.csv", index=False)
display(submission.head())
print(f"Rows: {len(submission):,}")
print(f"Contact rate: {submission['contact_customer'].mean():.1%}")


### Week 6 completion check

- Run the notebook from a fresh runtime from top to bottom.
- Validate `submission.csv` with `checks/validate_submission.py`.
- Inspect the output with the shared output explorer.
- Complete the one-page retention memo and three-minute methodology presentation.
- Include the required AI-use disclosure.


## Generative AI use disclosure

Complete the disclosure specified in `../../shared/ai_use_disclosure.md`. Record how AI was used, what you verified, and what decisions remain your own.
